In [16]:
%pip install ucimlrepo

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BrooklynSales").getOrCreate()

df_sales = spark.read.csv('/content/drive/MyDrive/brooklyn_sales_map.csv', header=True, inferSchema=True)

for col in df_sales.columns:
    df_sales = df_sales.withColumnRenamed(col, col.strip().replace(' ', '_').lower())

avg_gross_sqft = df_sales.select(F.avg("gross_sqft")).collect()[0][0]
print(f"Средняя общая площадь: {avg_gross_sqft:.2f}")

df_with_deviation = df_sales.select(
    "sale_price",
    (F.col("sale_price") - F.avg("sale_price").over(Window.partitionBy())).alias("price_deviation")
)
df_with_deviation.show(5)

df_avg_sqft_year = df_sales.groupBy("year_built").agg(F.avg("gross_sqft").alias("avg_sqft"))
df_avg_sqft_year.sort("year_built").show(5)

df_neighborhood_price = df_sales.groupBy("neighborhood", "building_class_category") \
    .agg(F.avg("sale_price").alias("avg_sale_price"))
df_neighborhood_price.show(5)

print(f"Количество строк до фильтрации: {df_sales.count()}")
df_filtered = df_sales.filter(F.col("year_built") <= 2000)
df_filtered = df_filtered.na.drop(how='all')
print(f"Количество строк после фильтрации: {df_filtered.count()}")


Средняя общая площадь: 2970.88
+------------+--------------------+
|  sale_price|     price_deviation|
+------------+--------------------+
|4.99401179E8|4.9889442452230716E8|
|      3.45E8|3.4449324552230716E8|
|       3.4E8|3.3949324552230716E8|
|   2.76947E8|2.7644024552230716E8|
|     2.025E8| 2.019932455223072E8|
+------------+--------------------+
only showing top 5 rows
+----------+------------------+
|year_built|          avg_sqft|
+----------+------------------+
|         0| 50.95310868929809|
|      1050|            3420.0|
|      1380|            3340.0|
|      1800|401.15875912408757|
|      1822|            3131.5|
+----------+------------------+
only showing top 5 rows
+-----------------+-----------------------+------------------+
|     neighborhood|building_class_category|    avg_sale_price|
+-----------------+-----------------------+------------------+
| BROOKLYN HEIGHTS|    22  STORE BUILDINGS|1748589.4137931035|
|          MIDWOOD|   05  TAX CLASS 1 V...|455718.3492063

In [26]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

spark = SparkSession.builder.appName("BankMarketingML").getOrCreate()

from ucimlrepo import fetch_ucirepo
bank_marketing = fetch_ucirepo(id=222)

pdf = pd.concat([bank_marketing.data.features, bank_marketing.data.targets], axis=1)

df = spark.createDataFrame(pdf)

categorical_cols = [c for c, t in df.dtypes if t == 'string' and c != 'y']

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categorical_cols]
label_indexer = StringIndexer(inputCol="y", outputCol="label")

feature_cols = [c+"_idx" for c in categorical_cols] + [c for c, t in df.dtypes if t != 'string' and c != 'y']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)


def print_metrics(predictions, model_name):
    print(f"\n=== Метрики для {model_name} ===")
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

    acc = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
    prec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
    rec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")

    print("Confusion Matrix:")
    predictions.groupBy("label", "prediction").count().sort("label", "prediction").show()

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

lr = LogisticRegression(labelCol="label", featuresCol="features")
grid_lr = ParamGridBuilder() \
    .addGrid(lr.maxIter, [10, 100]) \
    .addGrid(lr.regParam, [0.1, 0.5, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
grid_dt = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [3, 5, 9]) \
    .build()

rf = RandomForestClassifier(labelCol="label", featuresCol="features")
grid_rf = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [3, 5, 9]) \
    .addGrid(rf.numTrees, [5, 11, 25]) \
    .build()


models = [("Logistic Regression", lr, grid_lr),
          ("Decision Tree", dt, grid_dt),
          ("Random Forest", rf, grid_rf)]

for name, algo, grid in models:
    pipeline = Pipeline(stages=indexers + [label_indexer, assembler, algo])

    cv = CrossValidator(estimator=pipeline,
                        estimatorParamMaps=grid,
                        evaluator=evaluator,
                        numFolds=3)

    print(f"Обучение и поиск лучших параметров для {name}...")
    cv_model = cv.fit(train_data)
    results = cv_model.transform(test_data)

    print_metrics(results, name)

Обучение и поиск лучших параметров для Logistic Regression...

=== Метрики для Logistic Regression ===
Accuracy: 0.8855
Precision: 0.8618
Recall: 0.8855
Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7804|
|  0.0|       1.0|   40|
|  1.0|       0.0|  979|
|  1.0|       1.0|   79|
+-----+----------+-----+

Обучение и поиск лучших параметров для Decision Tree...

=== Метрики для Decision Tree ===
Accuracy: 0.9002
Precision: 0.8852
Recall: 0.9002
Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7661|
|  0.0|       1.0|  183|
|  1.0|       0.0|  705|
|  1.0|       1.0|  353|
+-----+----------+-----+

Обучение и поиск лучших параметров для Random Forest...

=== Метрики для Random Forest ===
Accuracy: 0.9010
Precision: 0.8862
Recall: 0.9010
Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7689|
|  0.0|  